In [ ]:
# import required Python modules

import os
import requests
from azure.identity import AzureCliCredential
from datetime import datetime, timedelta, timezone
import azure.storage.blob
from urllib.parse import urlparse
import yaml

In [ ]:
MPCPRO_APP_ID = "https://geocatalog.spatio.azure.com"
CONTAINER_URI = "https://mpcpstorageaccount.blob.core.windows.net/stac-items"
GEOCATALOG_URI = "https://geospatialdm.fmd9dgfcd2fab5hw.westeurope.geocatalog.spatio.azure.com"
API_VERSION = "2025-04-30-preview"

In [ ]:
# # parse the container URL
# parsed_url = urlparse(CONTAINER_URI)
# account_url = f"{parsed_url.scheme}://{parsed_url.netloc}"
# account_name = parsed_url.netloc.split(".")[0]
# container_name = parsed_url.path.lstrip("/")

# credential = azure.identity.AzureCliCredential()
# blob_service_client = azure.storage.blob.BlobServiceClient(
#     account_url=account_url,
#     credential=credential,
# )

# now = datetime.now(timezone.utc).replace(microsecond=0)
# key = blob_service_client.get_user_delegation_key(
#     key_start_time=now + timedelta(hours=-1),
#     key_expiry_time=now + timedelta(hours=1),
# )

# sas_token = azure.storage.blob.generate_container_sas(
#     account_name=account_name,
#     container_name=container_name,
#     user_delegation_key=key,
#     permission=azure.storage.blob.ContainerSasPermissions(
#         read=True,
#         list=True,
#     ),
#     start=now + timedelta(hours=-1),
#     expiry=now + timedelta(hours=4),
# )

In [ ]:
# obtain an access token
credential = AzureCliCredential()
access_token = credential.get_token(f"{MPCPRO_APP_ID}/.default")

In [ ]:
# # create POST payload for the ingestion source API
# # payload for the POST request
# payload = {
#     "Kind": "SasToken",
#     "connectionInfo": {
#         "containerUrl": CONTAINER_URI,
#         "sasToken": sas_token,
#     }
# }

In [ ]:
# # STAC Collection API endpoint
# endpoint = f"{GEOCATALOG_URI}/inma/ingestion-sources"

# # Make the POST request
# response = requests.post(
#     endpoint, 
#     json=payload,
#     headers={"Authorization": f"Bearer {access_token.token}"},
#     params={"api-version": API_VERSION},
# )

# # print the response
# if response.status_code == 201 or response.status_code == 200:
#     print("Ingestion source created successfully")
#     ingestion_source_id = response.json().get("id")
# else:
#     print(f"Failed to create ingestion: {response.text}")

In [ ]:
import os 
import requests
import yaml
from pprint import pprint
from azure.identity import AzureCliCredential
import pystac


MPCPRO_APP_ID = "https://geocatalog.spatio.azure.com"
GEOCATALOG_URI = "https://geospatialdm.fmd9dgfcd2fab5hw.westeurope.geocatalog.spatio.azure.com"
API_VERSION = "2025-04-30-preview"


collection_id = "Nigeria-CHIRPS"
collection_title = f"Nigeria CHIRPS Collection" 
collection_desc = f"Collection of CHIRPS Earth observation data"


# Create spatial extent
bbox = [2.316388, 3.837669, 15.126447, 14.153350]  # placeholder, replace with actual data at a later date
spatial_extent = pystac.SpatialExtent([bbox])

# Create temporal extent, use current date time or replace with existing datetimes in stac_item
start_datetime = datetime(1981, 1, 1, 0, 0, tzinfo=timezone.utc)
temporal_extent = pystac.TemporalExtent([[start_datetime, None]])
extent = pystac.Extent(spatial=spatial_extent, temporal=temporal_extent)

# Create the STAC Collection
collection = pystac.Collection(
    id=collection_id,
    description=collection_desc,
    extent=extent,
    title=collection_title,
    license="private",
)

# Add keywords and provider
collection.keywords = ["CHIRPS", "satellite", "weather", "UCSB"]
collection.providers = [
    pystac.Provider(
        name="UCSB",
        roles=["producer", "licensor"],
        url="https://data.chc.ucsb.edu"
    )
]

collection_dict = collection.to_dict()
collection_dict['stac_version'] = '1.0.0'
collection_dict['item_assets'] = {
    "data": {
        "type": "image/tiff; application=geotiff",
        "roles": ["data"],
        "title": "CHIRPS Precipitation Data",
        "raster:bands": [
            {
                "data_type": "float32",
                "nodata": -9999,
                "unit": "mm"
            }
        ]
    }
}

In [ ]:
# create a new collection with the collection api
response = requests.post(
    f"{GEOCATALOG_URI}/stac/collections",
    json=collection_dict,
    headers={"Authorization": "Bearer " + access_token.token},
    params={"api-version": API_VERSION},
)

if response.status_code == 201 or response.status_code == 200:
    print("Collection created successfully")
    pprint(response.json())
else:
    print(f"Failed to create ingestion: {response.text}")

# to-do: Better error handling and reporting; I think I have some code in the other files that handle this better

In [ ]:
# # code to build the `catalog.json` file and save it to blob storage under `stac-items/catalog.json`
# import json
# from azure.identity import AzureCliCredential
# from azure.storage.blob import BlobServiceClient

# # Configuration
# STORAGE_ACCOUNT_NAME = "mpcpstorageaccount"
# CONTAINER_NAME = "stac-items"
# ACCOUNT_URL = f"https://{STORAGE_ACCOUNT_NAME}.blob.core.windows.net"

# # Catalog metadata
# CATALOG_ID = "nigeria-chirps-catalog"
# CATALOG_TITLE = "Nigeria CHIRPS Data Catalog"
# CATALOG_DESCRIPTION = "CHIRPS v2.0 precipitation data for Nigeria (1981-2025)"
# STAC_VERSION = "1.0.0"

# # Output file
# OUTPUT_FILE = "catalog.json"

# print("Connecting to Azure Blob Storage...")

# # Authenticate using Azure CLI credentials
# credential = AzureCliCredential()
# blob_service_client = BlobServiceClient(
#     account_url=ACCOUNT_URL,
#     credential=credential
# )

# # Get container client
# container_client = blob_service_client.get_container_client(CONTAINER_NAME)

# print(f"Listing blobs in container '{CONTAINER_NAME}'...")

# # List all blobs and filter for STAC item JSON files
# links = []
# blob_count = 0

# for blob in container_client.list_blobs():
#     # Filter for JSON files matching the pattern
#     if blob.name.endswith('.json') and 'nigeria-cog-chirps-v2.0.' in blob.name:
#         blob_url = f"{ACCOUNT_URL}/{CONTAINER_NAME}/{blob.name}"
        
#         link = {
#             "rel": "item",
#             "href": blob_url,
#             "title": "Nigeria CHIRPS item",
#             "type": "application/json"
#         }
#         links.append(link)
#         blob_count += 1
        
#         # Progress indicator
#         if blob_count % 1000 == 0:
#             print(f"  Processed {blob_count} items...")

# print(f"\nFound {blob_count} STAC item JSON files")

# # Build the catalog
# catalog = {
#     "type": "Catalog",
#     "id": CATALOG_ID,
#     "title": CATALOG_TITLE,
#     "description": CATALOG_DESCRIPTION,
#     "stac_version": STAC_VERSION,
#     "links": links
# }

# # Write catalog to file
# print(f"\nWriting catalog to Azure Blob store...")
# container = CONTAINER_NAME
# blob_path = OUTPUT_FILE

# blob_client = blob_service_client.get_blob_client(
#         container=container,
#         blob=blob_path
#     )
# catalog = json.dumps(catalog, indent=2)
# blob_client.upload_blob(catalog.encode('utf-8'), overwrite=True)
# print(f"Uploaded STAC catalog JSON to {container_name}/{blob_path}")

In [ ]:
import os
import requests
import yaml
from azure.identity import AzureCliCredential

MPCPRO_APP_ID = "https://geocatalog.spatio.azure.com"
GEOCATALOG_URI = "https://geospatialdm.fmd9dgfcd2fab5hw.westeurope.geocatalog.spatio.azure.com"
API_VERSION = "2025-04-30-preview"

COLLECTION_ID = collection_id
catalog_href = "https://mpcpstorageaccount.blob.core.windows.net/stac-items/catalog.json"

skip_existing_items = False
keep_original_assets = False
timeout_seconds = 300

In [ ]:
# Obtain an access token
credential = AzureCliCredential()
access_token = credential.get_token(f"{MPCPRO_APP_ID}/.default")

In [ ]:
url = f"{GEOCATALOG_URI}/inma/collections/{COLLECTION_ID}/ingestions"
body = {
    "importType": "StaticCatalog",
    "sourceCatalogUrl": catalog_href,
    "skipExistingItems": skip_existing_items,
    "keepOriginalAssets": keep_original_assets,
}

In [ ]:
ing_response = requests.post(
    url,
    json=body,
    timeout=timeout_seconds,
    headers={"Authorization": f"Bearer {access_token.token}"},
    params={"api-version": API_VERSION},
)

In [ ]:
if ing_response.status_code == 201:
    print("Ingestion created successfully")
    ingestion_id = ing_response.json()["id"]
    print(f"Created ingestion with ID: {ingestion_id}")
else:
    print(f"Failed to create ingestion: {ing_response.text}")

In [ ]:
geocatalog_url = GEOCATALOG_URI
runs_endpoint = (
    f"{geocatalog_url}/inma/collections/{collection_id}/ingestions/{ingestion_id}/runs"
)

wf_response = requests.post(
    runs_endpoint,
    headers={"Authorization": f"Bearer {access_token.token}"},
    params={"api-version": API_VERSION},
)

if wf_response.status_code == 201:
    print("Workflow started successfully")
else:
    print(f"Failed to create ingestion run: {wf_response.text}")

In [ ]:
# del_is_endpoint = f"{GEOCATALOG_URI}/inma/ingestion-sources/{ingestion_source_id}"
# del_is_response = requests.delete(
#     del_is_endpoint,
#     headers={"Authorization": f"Bearer {access_token.token}"},
#     params={"api-version": API_VERSION},
# )

# if del_is_response.status_code == 200:
#     print("Ingestion source deleted successfully")
# else:
#     print(f"Failed to delete ingestion source")